# EEG Analysis — OpenNeuro ds001787 (Meditation)
**Main library:** MNE-Python  
**Pipeline:** BIDS loading → Preprocessing → Flat channel detection → Artifact rejection → PSD analysis  
**Scope:** All subjects in the dataset

> 💡 Every cell has detailed comments. Run them top to bottom in order.

---
## 0. Install dependencies
Run this cell only once.

In [ ]:
# Install required libraries (only needed once)
# !pip install mne mne-bids openneuro-py matplotlib numpy scipy pandas

---
## 1. Import libraries

In [ ]:
import mne
import mne_bids
import openneuro
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')

print(f"MNE version:      {mne.__version__}")
print(f"MNE-BIDS version: {mne_bids.__version__}")

---
## 2. Global parameters

All tunable parameters are defined here so you can adjust them in one place.

In [ ]:
# ═══════════════════════════════════════════════
#  GLOBAL PARAMETERS — edit here
# ═══════════════════════════════════════════════

BIDS_ROOT  = Path('./ds001787')   # local dataset folder
TASK_NAME  = 'meditation'         # BIDS task label (verify with print_dir_tree)

# Preprocessing
L_FREQ     = 1.0                  # band-pass high-pass cutoff (Hz)
H_FREQ     = 40.0                 # band-pass low-pass cutoff (Hz)

# Flat channel detection
FLAT_STD_THRESHOLD = 0.5e-6       # std  < 0.5 uV  -> likely flat
FLAT_PTP_THRESHOLD = 1.0e-6       # ptp  < 1.0 uV  -> likely flat

# Artifact rejection
EPOCH_DURATION = 2.0              # epoch length (s) for z-score rejection
Z_THRESHOLD    = 3.0              # z-score threshold

# PSD
PSD_FMIN   = 1.0
PSD_FMAX   = 40.0
N_FFT      = 2048
N_OVERLAP  = 512

# Frequency bands
BAND_DEFINITIONS = {
    'delta': (0.5,  4),
    'theta': (4,    8),
    'alpha': (8,   13),
    'beta':  (13,  30),
    'gamma': (30,  40),
}

print("Parameters set OK")

---
## 3. Download full dataset from OpenNeuro

In [ ]:
BIDS_ROOT.mkdir(exist_ok=True)

# Download the entire dataset (no include filter).
# This may take several minutes depending on dataset size.
openneuro.download(
    dataset='ds001787',
    target_dir=BIDS_ROOT
)

print("Download complete!")

---
## 4. Discover all subjects

In [ ]:
# Read participants.tsv to get the full subject list
participants_file = BIDS_ROOT / 'participants.tsv'
df_participants   = pd.read_csv(participants_file, sep='\t')

# Extract subject IDs — strip the 'sub-' prefix expected by MNE-BIDS
all_subjects = [
    p.replace('sub-', '') for p in df_participants['participant_id'].tolist()
]

print(f"Subjects found: {len(all_subjects)}")
print(f"IDs: {all_subjects}")
display(df_participants)

---
## 5. Pipeline functions

Each processing step is wrapped in a reusable function so the per-subject loop stays clean.

In [ ]:
# ──────────────────────────────────────────────────────
# 5a. LOAD raw data for one subject
# ──────────────────────────────────────────────────────
def load_subject(subject_id):
    """Load raw BIDS EEG data for a given subject ID string (e.g. '01')."""
    bids_path = mne_bids.BIDSPath(
        subject=subject_id,
        task=TASK_NAME,
        root=BIDS_ROOT,
        datatype='eeg'
    )
    return mne_bids.read_raw_bids(bids_path, verbose=False)


# ──────────────────────────────────────────────────────
# 5b. PREPROCESS: band-pass + average reference
# ──────────────────────────────────────────────────────
def preprocess(raw):
    """Apply band-pass filter and average re-reference."""
    rp = raw.copy().load_data()
    # Band-pass 1-40 Hz: the 40 Hz low-pass already suppresses 50 Hz line noise
    rp.filter(l_freq=L_FREQ, h_freq=H_FREQ, method='fir', verbose=False)
    rp.set_eeg_reference('average', projection=False, verbose=False)
    return rp


# ──────────────────────────────────────────────────────
# 5c. FLAT CHANNEL DETECTION: mark bads in info
# ──────────────────────────────────────────────────────
def detect_flat_channels(raw_prep):
    """
    Flag channels where std < FLAT_STD_THRESHOLD AND ptp < FLAT_PTP_THRESHOLD.
    Channels are marked as info['bads'] (not dropped) so they can be inspected.
    Returns list of flat channel names.
    """
    data, _ = raw_prep[:, :]
    ch_std   = data.std(axis=1)
    ch_ptp   = np.ptp(data, axis=1)

    flat_mask = (ch_std < FLAT_STD_THRESHOLD) & (ch_ptp < FLAT_PTP_THRESHOLD)
    flat_chs  = [raw_prep.ch_names[i] for i in np.where(flat_mask)[0]]

    if flat_chs:
        raw_prep.info['bads'] = list(set(raw_prep.info['bads'] + flat_chs))

    return flat_chs


# ──────────────────────────────────────────────────────
# 5d. ARTIFACT REJECTION: z-score on 2-s epochs
# ──────────────────────────────────────────────────────
def reject_artifacts(raw_prep):
    """
    Split signal into EPOCH_DURATION-s epochs, compute peak-to-peak z-score
    per channel, reject epochs where any channel exceeds Z_THRESHOLD.
    Channels in info['bads'] are excluded automatically.
    Returns (raw_clean, stats_dict).
    """
    good_picks = mne.pick_types(raw_prep.info, eeg=True, exclude='bads')
    data, _    = raw_prep[good_picks, :]
    info_clean = mne.pick_info(raw_prep.info, good_picks)

    sfreq         = raw_prep.info['sfreq']
    epoch_samples = int(EPOCH_DURATION * sfreq)
    n_ch, n_samp  = data.shape
    n_epochs      = n_samp // epoch_samples

    # Peak-to-peak amplitude per epoch per channel
    ptp = np.zeros((n_ch, n_epochs))
    for ep in range(n_epochs):
        s, e = ep * epoch_samples, (ep + 1) * epoch_samples
        ptp[:, ep] = data[:, s:e].max(axis=1) - data[:, s:e].min(axis=1)

    # Z-score across epochs, independently per channel
    mean_ptp = ptp.mean(axis=1, keepdims=True)
    std_ptp  = ptp.std(axis=1,  keepdims=True)
    std_ptp[std_ptp == 0] = 1e-10
    z_scores = (ptp - mean_ptp) / std_ptp

    # An epoch is rejected if at least one channel exceeds the threshold
    epochs_to_reject = np.any(z_scores > Z_THRESHOLD, axis=0)
    epochs_clean     = ~epochs_to_reject

    # Concatenate only clean epochs into a new Raw object
    segments = [
        data[:, ep * epoch_samples:(ep + 1) * epoch_samples]
        for ep in range(n_epochs) if epochs_clean[ep]
    ]
    data_clean = np.concatenate(segments, axis=1)
    raw_clean  = mne.io.RawArray(data_clean, info_clean, verbose=False)

    stats = {
        'n_epochs_total':    n_epochs,
        'n_epochs_rejected': int(epochs_to_reject.sum()),
        'pct_rejected':      round(100 * epochs_to_reject.sum() / n_epochs, 1),
        'duration_clean_s':  round(data_clean.shape[1] / sfreq, 1),
    }
    return raw_clean, stats


# ──────────────────────────────────────────────────────
# 5e. COMPUTE PSD and extract band power
# ──────────────────────────────────────────────────────
def compute_band_power(raw_clean):
    """
    Compute Welch PSD on the clean signal.
    Returns (psd_object, freqs, psd_data, band_power_dict).
    """
    psd      = raw_clean.compute_psd(
        method='welch',
        fmin=PSD_FMIN, fmax=PSD_FMAX,
        n_fft=N_FFT, n_overlap=N_OVERLAP
    )
    psd_data = psd.get_data()   # shape: (n_ch, n_freqs)
    freqs    = psd.freqs

    band_powers = {}
    for band, (fmin, fmax) in BAND_DEFINITIONS.items():
        idx = (freqs >= fmin) & (freqs <= fmax)
        mean_power = psd_data[:, idx].mean()          # mean over channels and freqs
        band_powers[f'{band}_dB'] = round(10 * np.log10(mean_power), 3)

    return psd, freqs, psd_data, band_powers


print("Pipeline functions defined OK")

---
## 6. Run pipeline on ALL subjects

For every subject:
1. **Load** raw BIDS data
2. **Preprocess** (band-pass 1–40 Hz + average reference)
3. **Flat channel detection** (mark bads)
4. **Artifact rejection** (z-score on 2-s epochs)
5. **PSD + band power** (Welch)

Results are stored in `results_df` (one row per subject) and in `psd_store` (PSD objects).

In [ ]:
results_rows = []   # accumulates one dict per subject
psd_store    = {}   # subject_id -> {'freqs', 'psd_data', 'raw_psd_obj', 'clean_psd_obj'}
failed_subs  = []   # subjects that raised an exception

print(f"Processing {len(all_subjects)} subject(s) ...\n")
print(f"{'Sub':>6}  {'Flat':>5}  {'Rej%':>6}  {'Clean(s)':>9}  {'Alpha(dB)':>10}")
print('-' * 46)

for sub in all_subjects:
    try:
        # 1. Load
        raw = load_subject(sub)

        # 2. Preprocess
        raw_prep = preprocess(raw)

        # 3. Flat channel detection
        flat_chs = detect_flat_channels(raw_prep)

        # 4. Artifact rejection
        raw_clean, rej_stats = reject_artifacts(raw_prep)

        # 5. PSD + band power on clean signal
        psd_obj, freqs, psd_data, band_powers = compute_band_power(raw_clean)

        # Also compute raw PSD (wider range) for the before/after comparison
        psd_raw_obj = raw.copy().load_data().compute_psd(
            method='welch', fmin=0.5, fmax=80.0,
            n_fft=N_FFT, n_overlap=N_OVERLAP
        )

        # Store PSD objects
        psd_store[sub] = {
            'freqs':         freqs,
            'psd_data':      psd_data,
            'raw_psd_obj':   psd_raw_obj,
            'clean_psd_obj': psd_obj,
        }

        # Accumulate row
        row = {
            'subject':        sub,
            'n_channels':     len(raw.ch_names),
            'n_flat':         len(flat_chs),
            'flat_channels':  ', '.join(flat_chs) if flat_chs else 'none',
            'duration_raw_s': round(raw.times[-1], 1),
            **rej_stats,
            **band_powers,
        }
        results_rows.append(row)

        print(f"  {sub:>4}  {len(flat_chs):>5}  {rej_stats['pct_rejected']:>5.1f}%  "
              f"{rej_stats['duration_clean_s']:>9.1f}  {band_powers['alpha_dB']:>10.2f}")

    except Exception as exc:
        print(f"  {sub:>4}  FAILED: {exc}")
        failed_subs.append(sub)

results_df = pd.DataFrame(results_rows)
print(f"\n{'─'*46}")
print(f"Done.  Successful: {len(results_rows)}   Failed: {len(failed_subs)}")
if failed_subs:
    print(f"Failed subjects: {failed_subs}")

---
## 7. Results table

In [ ]:
# Full per-subject results table
display(results_df)

# Save to CSV for downstream analysis
results_df.to_csv('results_all_subjects.csv', index=False)
print("Saved to 'results_all_subjects.csv'")

In [ ]:
# Descriptive statistics for band power across subjects
band_cols = [c for c in results_df.columns if c.endswith('_dB')]
print("--- Band power descriptive statistics (across subjects) ---")
display(results_df[band_cols].describe().round(2))

---
## 8. PSD before vs after processing — grand average

Grand-average PSD across all subjects: raw signal (left) vs clean signal (right).
Both subplots share the same y-axis for a fair visual comparison.

In [ ]:
# Interpolate every subject's PSD onto a common frequency axis
FMIN_PLOT, FMAX_PLOT = 0.5, 80.0
common_freqs = np.linspace(FMIN_PLOT, FMAX_PLOT, 500)

raw_means_all   = []
clean_means_all = []

for sub, store in psd_store.items():
    rp = store['raw_psd_obj']
    raw_db = 10 * np.log10(rp.get_data()).mean(axis=0)
    raw_means_all.append(np.interp(common_freqs, rp.freqs, raw_db))

    cp = store['clean_psd_obj']
    clean_db = 10 * np.log10(cp.get_data()).mean(axis=0)
    clean_means_all.append(np.interp(common_freqs, cp.freqs, clean_db))

raw_stack   = np.array(raw_means_all)    # (n_subs, n_freqs)
clean_stack = np.array(clean_means_all)

grand_raw_mean,   grand_raw_std   = raw_stack.mean(0),   raw_stack.std(0)
grand_clean_mean, grand_clean_std = clean_stack.mean(0), clean_stack.std(0)

# ── Plot ──
band_spans = [
    (0.5,  4,  '#e8d5f5', 'Delta'),
    (4,    8,  '#d5e8f5', 'Theta'),
    (8,   13,  '#d5f5e3', 'Alpha'),
    (13,  30,  '#f5f0d5', 'Beta'),
    (30,  40,  '#f5d5d5', 'Gamma'),
]

y_min = min(grand_raw_mean.min(), grand_clean_mean.min()) - 5
y_max = max(grand_raw_mean.max(), grand_clean_mean.max()) + 5
n_ok  = len(psd_store)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, mean, std, title, color in [
    (axes[0], grand_raw_mean,   grand_raw_std,   f'RAW signal (n={n_ok})',   '#c0392b'),
    (axes[1], grand_clean_mean, grand_clean_std, f'CLEAN signal (n={n_ok})', '#2980b9'),
]:
    for f_lo, f_hi, c, label in band_spans:
        ax.axvspan(f_lo, f_hi, alpha=0.2, color=c)
        ax.text((f_lo + f_hi) / 2, y_min + 1, label,
                ha='center', va='bottom', fontsize=7, color='#555')

    ax.plot(common_freqs, mean, color=color, lw=2, label='Grand mean')
    ax.fill_between(common_freqs, mean - std, mean + std,
                    alpha=0.2, color=color, label='+-1 SD')
    ax.axvline(50, color='grey', lw=1, linestyle=':', alpha=0.6, label='50 Hz ref')
    ax.set_xlim(FMIN_PLOT, FMAX_PLOT)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel('Frequency (Hz)', fontsize=11)
    ax.set_ylabel('Power (dB)', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold', color=color)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(f'PSD before vs after processing — grand average (n={n_ok})\nds001787',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('psd_before_after_grandavg.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'psd_before_after_grandavg.png'")

---
## 9. Individual PSD curves — all subjects overlaid

In [ ]:
# One line per subject on the clean PSD — useful to spot outliers
fig, ax = plt.subplots(figsize=(13, 5))

cmap       = plt.cm.tab20
sub_colors = [cmap(i / max(len(psd_store) - 1, 1)) for i in range(len(psd_store))]

for (sub, store), color in zip(psd_store.items(), sub_colors):
    cp   = store['clean_psd_obj']
    mean = 10 * np.log10(cp.get_data()).mean(axis=0)
    ax.plot(cp.freqs, mean, lw=1.2, alpha=0.75, color=color, label=f'sub-{sub}')

# Grand mean on top
ax.plot(common_freqs, grand_clean_mean, color='black', lw=2.5,
        linestyle='--', label='Grand mean', zorder=10)

for f_lo, f_hi, c, label in band_spans:
    ax.axvspan(f_lo, f_hi, alpha=0.12, color=c)

ax.set_xlim(PSD_FMIN, PSD_FMAX)
ax.set_xlabel('Frequency (Hz)', fontsize=11)
ax.set_ylabel('Power (dB)', fontsize=11)
ax.set_title(f'Clean PSD — individual subjects + grand mean (n={n_ok})', fontsize=12)
ax.legend(fontsize=7, ncol=4, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('psd_all_subjects.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'psd_all_subjects.png'")

---
## 10. Band power across subjects

In [ ]:
# Grouped bar chart: one cluster per subject, one bar per band
band_cols   = [c for c in results_df.columns if c.endswith('_dB')]
band_names  = [c.replace('_dB', '') for c in band_cols]
n_subs      = len(results_df)
n_bands     = len(band_cols)
x           = np.arange(n_subs)
width       = 0.15
band_colors = ['#9b59b6', '#3498db', '#2ecc71', '#f39c12', '#e74c3c']

fig, ax = plt.subplots(figsize=(max(10, n_subs * 1.2), 5))

for i, (col, bname, color) in enumerate(zip(band_cols, band_names, band_colors)):
    offset = (i - n_bands / 2 + 0.5) * width
    ax.bar(x + offset, results_df[col], width,
           label=bname.capitalize(), color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f'sub-{s}' for s in results_df['subject']], rotation=45, ha='right')
ax.set_ylabel('Mean power (dB)', fontsize=11)
ax.set_title('Band power per subject — clean signal', fontsize=12)
ax.legend(title='Band', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('band_power_all_subjects.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'band_power_all_subjects.png'")

In [ ]:
# Box plot: distribution of each band power across subjects
fig, ax = plt.subplots(figsize=(8, 5))

data_for_box = [results_df[col].values for col in band_cols]
bp = ax.boxplot(data_for_box, patch_artist=True)

for patch, color in zip(bp['boxes'], band_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels([b.capitalize() for b in band_names])
ax.set_ylabel('Mean power (dB)', fontsize=11)
ax.set_title('Distribution of band power across subjects', fontsize=12)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('band_power_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'band_power_boxplot.png'")

---
## 11. QC — artifact rejection rate per subject

In [ ]:
# Green < 20%, orange 20-40%, red > 40%
bar_colors = [
    '#e74c3c' if p > 40 else '#f39c12' if p > 20 else '#2ecc71'
    for p in results_df['pct_rejected']
]

fig, ax = plt.subplots(figsize=(max(8, n_subs * 0.8), 4))

ax.bar(range(n_subs), results_df['pct_rejected'], color=bar_colors, edgecolor='white')
ax.axhline(40, color='red',    lw=1.5, linestyle='--', label='40% warning')
ax.axhline(20, color='orange', lw=1.5, linestyle='--', label='20% caution')

for i, pct in enumerate(results_df['pct_rejected']):
    ax.text(i, pct + 0.5, f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(range(n_subs))
ax.set_xticklabels([f'sub-{s}' for s in results_df['subject']], rotation=45, ha='right')
ax.set_ylabel('Epochs rejected (%)', fontsize=11)
ax.set_title('Artifact rejection rate per subject', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, max(results_df['pct_rejected'].max() + 10, 50))

plt.tight_layout()
plt.savefig('qc_rejection_rate.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'qc_rejection_rate.png'")

---
## 12. Summary and next steps

**What we did:**
- Downloaded and loaded **all subjects** from the BIDS dataset
- Per-subject pipeline: band-pass 1-40 Hz -> average reference -> flat channel detection (marked as bads) -> z-score artifact rejection on 2-s epochs
- Welch PSD + band power extraction (delta / theta / alpha / beta / gamma) for every subject
- Grand-average PSD before vs after processing (2 subplots, shared y-axis)
- Individual PSD overlay and band power distributions
- QC chart of rejection rate per subject
- All results exported to `results_all_subjects.csv`

**Possible next steps:**
- **ICA**: remove residual ocular/cardiac artifacts with Independent Component Analysis
- **Statistical testing**: t-test or ANOVA on alpha power across groups
- **Epoching**: segment signal into event-locked windows if events are present
- **Time-frequency**: how does alpha power evolve during the session?
- **Coherence**: how synchronised are different brain regions across subjects?